# Gradient norms and entropy loss terms: compact interactive report

Copy of the compact gradient-norm workflow for `20260725_164256_resnet50_aig_plus_neg_entropy_coef_lambda1em4_120ep_repeats3_ordered`, with muted Plotly traces and one legend item per run so repeated runs can be hidden independently.


In [ ]:
from pathlib import Path
import os
import sys
import tempfile

import numpy as np
import pandas as pd
from IPython.display import display

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "src" / "net_complexity").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from net_complexity.studies import (
    gradient_norm_catalog,
    load_study,
    plot_channel_counts,
    plot_gradient_norms,
    plot_metric,
)


In [ ]:
# The only required setting. This can point to either a study directory or one run directory.
STUDY_NAME = "20260725_164256_resnet50_aig_plus_neg_entropy_coef_lambda1em4_120ep_repeats3_ordered"
STUDY_DIR = (REPO_ROOT / "outputs" / "studies" / STUDY_NAME).resolve()

summary_df, history_df = load_study(STUDY_DIR)
summary_df = summary_df.copy()
history_df = history_df.copy()

print(f"study: {STUDY_DIR}")
print(f"runs: {history_df['run_name'].nunique()}, history: {history_df.shape}")
display(summary_df)


In [ ]:
# Muted interactive styling. Keep labels unique by run, while preserving beta in the text.
TRACE_OPACITY = 0.55
MUTED_COLORWAY = [
    "#4C78A8", "#9C755F", "#59A14F", "#B07AA1", "#F28E2B", "#76B7B2",
    "#E15759", "#79706E", "#8CD17D", "#BAB0AC", "#A0CBE8", "#D4A6C8",
]

def _format_run_label(row: pd.Series) -> str:
    label = row.get("run_label")
    run_name = row.get("run_name")
    if pd.isna(label) or label is None:
        return str(run_name)
    label = str(label)
    run_name = str(run_name)
    if label == run_name:
        return run_name
    return f"{label} | {run_name}"

run_display_by_name = summary_df.set_index("run_name").apply(_format_run_label, axis=1)
history_df["run_display_label"] = history_df["run_name"].map(run_display_by_name).fillna(history_df["run_name"])
summary_df["run_display_label"] = summary_df["run_name"].map(run_display_by_name).fillna(summary_df["run_name"])

def style_figure(fig, *, title: str | None = None):
    fig.update_layout(
        template="plotly_white",
        colorway=MUTED_COLORWAY,
        hovermode="x unified",
        legend={
            "x": 1.02,
            "y": 1.0,
            "xanchor": "left",
            "yanchor": "top",
            "groupclick": "togglegroup",
            "itemclick": "toggle",
            "itemdoubleclick": "toggleothers",
            "title": {"text": "Runs"},
        },
        margin={"r": 420, "t": 60, "b": 45, "l": 70},
    )
    if title is not None:
        fig.update_layout(title=title)

    color_by_group = {}
    for index, trace in enumerate(fig.data):
        group = trace.legendgroup or trace.name or f"trace-{index}"
        if not trace.legendgroup:
            trace.legendgroup = group
        if group not in color_by_group:
            color_by_group[group] = MUTED_COLORWAY[len(color_by_group) % len(MUTED_COLORWAY)]
        trace.opacity = TRACE_OPACITY
        if getattr(trace, "mode", None) == "lines":
            trace.line.width = 1.8
            trace.line.color = color_by_group[group]
        if trace.showlegend is None:
            trace.showlegend = True
    return fig

def show_metric(metric: str, *, title: str | None = None, yscale: str = "linear"):
    if metric not in history_df.columns:
        available = [col for col in history_df.columns if metric.split("_")[0] in col]
        raise ValueError(f"Column not found: {metric}. Related columns: {available}")
    fig = plot_metric(
        history_df,
        metric,
        label_col="run_display_label",
        yscale=yscale,
        interactive=True,
        show=False,
        title=title or f"{metric} / epoch",
    )
    style_figure(fig, title=title or f"{metric} / epoch")
    display(fig)
    return fig

def show_channel_counts():
    fig = plot_channel_counts(history_df, interactive=True, show=False)
    style_figure(fig, title="open / closed blocks by run")
    display(fig)
    return fig

def show_gradient_group(parameter_group: str = "total", statistic: str = "mean"):
    figures = plot_gradient_norms(
        history_df,
        parameter_group=parameter_group,
        statistic=statistic,
        yscale="log",
        interactive=True,
        show=False,
        label_col="run_display_label",
    )
    for metric, fig in figures.items():
        style_figure(fig, title=f"{metric} / epoch")
        display(fig)
    return figures


In [ ]:
catalog = gradient_norm_catalog(history_df)
if catalog.empty:
    raise ValueError("No grad_norm_* columns in history.csv. Check gradient_norm_logging.enabled.")

display(catalog)


In [ ]:
show_metric("valid_accuracy", title="valid_accuracy / epoch")


In [ ]:
show_channel_counts()


In [ ]:
show_metric("lambda_coef", title="lambda_coef / epoch", yscale="log")


In [ ]:
show_gradient_group(parameter_group="total", statistic="mean")


## Entropy and gate-loss terms

The run logs the total regularization gradient as `grad(output.loss - output.ce_loss)`. The scalar cells below split the logged loss contribution into the entropy term and the lambda-scaled gate term when the required columns are present.


In [ ]:
# Derived loss-term columns used by the following separate plots.
def _first_existing(df: pd.DataFrame, candidates: tuple[str, ...]) -> str | None:
    return next((col for col in candidates if col in df.columns), None)

beta_col = _first_existing(
    summary_df,
    (
        "model.entropy_regularization_coef",
        "label_beta",
        "label_entropy_coef",
        "mlflow.tags.entropy_regularization_coef",
    ),
)
if beta_col is None:
    raise ValueError("Cannot find entropy beta column in summary_df")

mode_col = _first_existing(
    summary_df,
    (
        "model.entropy_regularization",
        "label_entropy",
        "mlflow.tags.entropy_regularization",
    ),
)

beta_by_run = pd.to_numeric(summary_df.set_index("run_name")[beta_col], errors="coerce")
history_df["entropy_beta"] = history_df["run_name"].map(beta_by_run)

if mode_col is not None:
    mode_by_run = summary_df.set_index("run_name")[mode_col].astype(str)
    history_df["entropy_mode"] = history_df["run_name"].map(mode_by_run)
else:
    history_df["entropy_mode"] = "plus_negative_entropy"

entropy_sign_by_mode = {
    "disabled": 0.0,
    "plus_negative_entropy": 1.0,
    "minus_negative_entropy": -1.0,
}
history_df["entropy_sign"] = history_df["entropy_mode"].map(entropy_sign_by_mode).fillna(1.0)
lambda_values = pd.to_numeric(history_df["lambda_coef"], errors="coerce")

if "valid_negative_entropy" in history_df.columns:
    negative_entropy = pd.to_numeric(history_df["valid_negative_entropy"], errors="coerce")
    history_df["valid_entropy"] = -negative_entropy
    history_df["valid_entropy_loss_term"] = (
        history_df["entropy_sign"] * history_df["entropy_beta"] * negative_entropy
    )

if "valid_regularization_loss" in history_df.columns:
    regularization_loss = pd.to_numeric(history_df["valid_regularization_loss"], errors="coerce")
    history_df["valid_lambda_regularization_loss"] = lambda_values * regularization_loss

if "valid_mean_p_open" in history_df.columns:
    mean_p_open = pd.to_numeric(history_df["valid_mean_p_open"], errors="coerce")
    history_df["valid_lambda_mean_p_open_loss"] = lambda_values * mean_p_open

if {"valid_entropy_loss_term", "valid_lambda_mean_p_open_loss"}.issubset(history_df.columns):
    history_df["valid_gate_loss_from_terms"] = (
        history_df["valid_lambda_mean_p_open_loss"] + history_df["valid_entropy_loss_term"]
    )

derived_cols = [
    col for col in (
        "entropy_beta",
        "entropy_mode",
        "valid_entropy",
        "valid_entropy_loss_term",
        "valid_lambda_regularization_loss",
        "valid_lambda_mean_p_open_loss",
        "valid_gate_loss_from_terms",
        "valid_reg_loss",
    )
    if col in history_df.columns
]
display(history_df[["run_name", "epoch", *derived_cols]].head())


In [ ]:
show_metric(
    "valid_entropy_loss_term",
    title="entropy term in loss: sign * beta * negative_entropy / epoch",
)


In [ ]:
show_metric(
    "valid_lambda_regularization_loss",
    title="lambda * valid_regularization_loss / epoch",
    yscale="log",
)


In [ ]:
show_metric(
    "valid_lambda_mean_p_open_loss",
    title="lambda * valid_mean_p_open / epoch",
    yscale="log",
)


In [ ]:
show_metric(
    "valid_gate_loss_from_terms",
    title="lambda * mean_p_open + entropy term / epoch",
)


In [ ]:
show_metric(
    "valid_reg_loss",
    title="logged valid_reg_loss / epoch",
)


In [ ]:
# This is the logged gradient of the whole regularization part of the loss.
# It includes both the lambda-scaled gate term and the entropy term.
show_metric(
    "grad_norm_regularization_total_mean",
    title="grad(output.loss - output.ce_loss), total parameters / epoch",
    yscale="log",
)
